# ARTI308 – Lab 5: Feature Engineering (Classification)  

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the dataset

In [ ]:
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)
df.head(10)


Each row represents one person.  
The target variable is **income**, which indicates whether a person earns `<=50K` or `>50K`.

## 3. Quick dataset checks

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count").T)

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
print("Unique target classes:")
print(df["income"].value_counts())


The dataset does not use standard missing values heavily, but some columns may contain `?`, which represent unknown categories.  
We will handle them during preprocessing.

## 4. Target variable and class balance

In [ ]:
target_col = "income"

plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title("Income Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.show()


The classes are not perfectly balanced, so we should look at more than accuracy when evaluating the model.

## 5. Basic preprocessing before feature engineering

In [ ]:
df_fe = df.copy()

# Replace '?' with a readable category
df_fe.replace("?", "Unknown", inplace=True)

df_fe.head()


## 6. Feature engineering

### 6.1 Age groups  
Convert the numeric age into broader categories to capture life-stage patterns.

In [ ]:
df_fe["age_group"] = pd.cut(
    df_fe["age"],
    bins=[0, 25, 35, 50, 100],
    labels=["young", "adult", "middle_age", "senior"]
)

df_fe[["age", "age_group"]].head(10)


### 6.2 Work-hour groups  
Group weekly working hours into categories.

In [ ]:
df_fe["hours_group"] = pd.cut(
    df_fe["hours-per-week"],
    bins=[0, 25, 40, 60, 100],
    labels=["part_time", "full_time", "overtime", "extreme"]
)

df_fe[["hours-per-week", "hours_group"]].head(10)


### 6.3 Net capital feature  
Combine capital gain and capital loss into one feature.

In [ ]:
df_fe["net_capital"] = df_fe["capital-gain"] - df_fe["capital-loss"]
df_fe["has_capital_gain"] = (df_fe["capital-gain"] > 0).astype(int)

df_fe[["capital-gain", "capital-loss", "net_capital", "has_capital_gain"]].head(10)


### 6.4 Work intensity  
Create a feature that combines education level and working hours.

In [ ]:
df_fe["work_intensity"] = df_fe["education-num"] * df_fe["hours-per-week"]
df_fe[["education-num", "hours-per-week", "work_intensity"]].head(10)


### 6.5 Simplify marital status  
Reduce category complexity by grouping similar values.

In [ ]:
married_values = ["Married-civ-spouse", "Married-AF-spouse", "Married-spouse-absent"]

df_fe["marital_group"] = np.where(df_fe["marital-status"].isin(married_values), "Married", "Not_Married")

df_fe[["marital-status", "marital_group"]].head(10)


### 6.6 Reduce high-cardinality country categories  
Keep only the most frequent countries and group the rest into `Other`.

In [ ]:
top_k = 10
top_countries = df_fe["native-country"].value_counts().head(top_k).index
df_fe["native_country_reduced"] = np.where(
    df_fe["native-country"].isin(top_countries),
    df_fe["native-country"],
    "Other"
)

print("Unique native-country:", df_fe["native-country"].nunique())
print("Unique native_country_reduced:", df_fe["native_country_reduced"].nunique())
df_fe[["native-country", "native_country_reduced"]].head(10)


## 7. Prepare features for modeling

We will drop the original `native-country` after reducing it, and keep both original and engineered features where useful.

In [ ]:
drop_cols = ["income", "native-country"]

X = df_fe.drop(columns=drop_cols)
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


## 8. Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


## 9. Encoding and baseline model (Random Forest)

Machine learning models need numerical input, so categorical variables must be encoded.  
We use **One-Hot Encoding** for categorical columns and **Random Forest** as the baseline classifier.

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

model


## 10. Train the model and evaluate

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


Accuracy gives an overall measure, but the classification report gives more detail through precision, recall, and F1-score.

## 11. Feature importance

Random Forest can show which features contributed most to the prediction.

In [ ]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else np.array([])
all_feature_names = np.concatenate([cat_feature_names, np.array(numeric_cols)])

importances = model.named_steps["rf"].feature_importances_

fi = (pd.DataFrame({"feature": all_feature_names, "importance": importances})
        .sort_values("importance", ascending=False))

fi.head(15)


In [ ]:
plt.figure(figsize=(8,5))
top_n = 15
sns.barplot(data=fi.head(top_n), x="importance", y="feature")
plt.title(f"Top {top_n} Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


This chart highlights which original and engineered features were the most useful for predicting income.